# Single Model Baseline Test

This notebook tests the evaluation pipeline with ONE embedding model (text-embedding-3-small) to:
1. Validate the RAGFlow integration
2. Verify Ragas evaluation works end-to-end
3. Establish a baseline for comparison
4. Debug any issues before scaling to multiple models

In [ ]:
# Setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

import pandas as pd
from ragflow_client import RAGFlowClient
from evaluation_pipeline import EvaluationPipeline
from utils import load_dataset_from_jsonl
from config import EMBEDDING_MODELS

print("✓ Imports successful")

## 1. Load Test Dataset

In [ ]:
# Load dataset
dataset_path = Path.cwd().parent / 'data' / 'test_qa_pairs.jsonl'
dataset = load_dataset_from_jsonl(str(dataset_path))

print(f"Loaded {len(dataset)} test questions")
print("\nFirst 3 questions:")
for i, item in enumerate(dataset[:3]):
    print(f"{i+1}. {item['user_input']}")

## 2. Initialize RAGFlow Client

In [ ]:
# Initialize RAGFlow client
client = RAGFlowClient()

# Test connection
print("Testing RAGFlow connection...")
test_result = client.query("Test query", model_name="default")
if test_result['success']:
    print("✓ RAGFlow connection successful")
else:
    print(f"✗ Connection failed: {test_result['error']}")

## 3. Select Baseline Model (OpenAI text-embedding-3-small)

In [ ]:
# Get the first model (OpenAI small)
baseline_model = EMBEDDING_MODELS[0]  # text-embedding-3-small

print("Baseline Model:")
print("=" * 60)
for key, value in baseline_model.items():
    print(f"{key}: {value}")
print("=" * 60)

## 4. Run Evaluation Pipeline

In [ ]:
# Initialize evaluation pipeline
pipeline = EvaluationPipeline(
    ragflow_client=client,
    evaluator_model='gpt-4o-mini'  # LLM for Ragas metrics
)

print("✓ Evaluation pipeline initialized")
print(f"✓ Ragas evaluator: gpt-4o-mini")

In [ ]:
# Run evaluation
print("\n" + "="*70)
print("STARTING EVALUATION")
print("="*70)

result = pipeline.run_evaluation(
    test_dataset=dataset,
    model_config=baseline_model,
    verbose=True
)

## 5. Analyze Results

In [ ]:
# Check if evaluation succeeded
if result['success']:
    print("✅ Evaluation completed successfully!\n")
    
    # Get results as DataFrame
    results_df = result['ragas_results'].to_pandas()
    
    # Display detailed results
    print("\nDetailed Results (per question):")
    print("=" * 80)
    print(results_df.to_string())
    print("=" * 80)
    
    # Summary statistics
    print("\nSummary Statistics:")
    print("=" * 80)
    summary = results_df.mean().to_frame(name='Mean')
    summary['Std'] = results_df.std()
    summary['Min'] = results_df.min()
    summary['Max'] = results_df.max()
    print(summary.to_string())
    print("=" * 80)
    
else:
    print("✗ Evaluation failed!")
    if 'error' in result:
        print(f"Error: {result['error']}")

# Show failed queries if any
if result['failed_queries']:
    print(f"\n⚠️  {len(result['failed_queries'])} queries failed:")
    for i, failed in enumerate(result['failed_queries'][:3]):
        print(f"{i+1}. {failed['question'][:60]}...")
        print(f"   Error: {failed['error']}")

## 6. Inspect Individual Responses

In [ ]:
# Look at a few examples
if result['success'] and result['eval_data']:
    print("\nExample Responses:\n")
    
    for i, item in enumerate(result['eval_data'][:3]):
        print("=" * 80)
        print(f"Question {i+1}: {item['user_input']}")
        print(f"\nGenerated Answer:\n{item['response']}")
        print(f"\nGround Truth:\n{item['reference']}")
        print(f"\nRetrieved Contexts: {len(item['retrieved_contexts'])}")
        if item['retrieved_contexts']:
            print(f"First context preview:\n{item['retrieved_contexts'][0][:200]}...")
        print("=" * 80 + "\n")

## 7. Visualize Metrics

In [ ]:
import plotly.graph_objects as go

if result['success']:
    # Create bar chart of average metrics
    metrics = results_df.mean()
    
    fig = go.Figure(data=[
        go.Bar(
            x=metrics.index,
            y=metrics.values,
            text=[f"{v:.3f}" for v in metrics.values],
            textposition='auto',
        )
    ])
    
    fig.update_layout(
        title=f"Baseline Performance: {baseline_model['display_name']}",
        xaxis_title="Metric",
        yaxis_title="Score",
        yaxis_range=[0, 1],
        height=500
    )
    
    fig.show()

## 8. Save Results

In [ ]:
from datetime import datetime
from utils import save_results

if result['success']:
    # Create results directory
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_dir = Path.cwd().parent / 'results' / f'baseline_{timestamp}'
    
    # Save
    save_results(
        results=result['ragas_results'],
        model_name=baseline_model['name'],
        output_dir=str(results_dir)
    )
    
    print(f"\n✅ Results saved to: {results_dir}")

## Summary

✅ If this notebook ran successfully, you have:
1. Validated RAGFlow integration
2. Confirmed Ragas evaluation works
3. Established baseline performance metrics
4. Identified any data format issues

**Next Step:** Run `02_multi_model_eval.ipynb` to compare all embedding models!